# EXPLANATIONS!

In [ ]:
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import torch
import yaml

from resnet_18 import make_resnet18
from bcos.modules.bcosconv2d import BcosConv2d
from functools import partial
from jsonschema import validate, ValidationError
from logging.handlers import RotatingFileHandler

from audio_dataset import CatDogAudioDataset, LABEL_MAP, REVERSE_LABEL_MAP
from custom_logger_formatter import CustomLoggerFormatter
from config.config_validation_template import CONFIG_TEMPLATE
from main import DATASET_MAPPING
from tune import TUNABLE_PARAMS

### 0. Config & setup

In [ ]:
EXPERIMENT_DIR = r"output\06-06-2026--14-48" # directory to the specific experiment

BEST_MODEL_DIR = r"job_0" # folder containing .pt model, relative from `EXPERIMENT_DIR`

DEVICE = "cuda"

In [ ]:
# Initialise Logger.
def _make_stream_handler(level: int) -> logging.StreamHandler:
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(CustomLoggerFormatter())
    return ch

level: int=logging.DEBUG
logger = logging.getLogger("test logger")
logger.setLevel(level)
logger.propagate = False  
ch = _make_stream_handler(level)
logger.addHandler(ch)
f_ch = RotatingFileHandler(f"{EXPERIMENT_DIR}/explanations.log")
f_ch.setLevel(level)
f_ch.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
)
logger.addHandler(f_ch)

logger.info(f"This file logs the testing process.")

In [ ]:
# validate the provided config file.
with open(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/run_config.yml", 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(f"{EXPERIMENT_DIR}/config.yml", 'r') as stream:
    general_config = yaml.safe_load(stream)

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

for tunable_param in TUNABLE_PARAMS.keys():
    if type(CONFIG[tunable_param]) == list:
        CONFIG[tunable_param] = CONFIG[tunable_param][0]

logger.info("config: {CONFIG}")

### 1. Load data


In [ ]:
dataset = CatDogAudioDataset(
    data_dirs=DATASET_MAPPING["train"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
logger.debug(f"Dataset size: {len(dataset)}")
logger.debug(f"Shape of first x element: {dataset[0][0].shape}")
logger.debug(f"First y element: {dataset[0][1]}")
test_data = CatDogAudioDataset(
    data_dirs=DATASET_MAPPING["test"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
logger.debug(f"Test dataset size: {len(test_data)}")

# Normalise
logger.debug("Fitting normalisation.")
dataset.fit_normalisation(list(range(len(dataset))))
test_data.mean = dataset.mean
test_data.std = dataset.std
logger.debug(
    "Normalisation fitted: "
    f"mean={dataset.mean}, std={dataset.std}"
)

In [ ]:
# upload your own dataset
# test_data = CatDogAudioDataset(
#     data_dirs="data/personal_recordings/",
#     target_sr=CONFIG["sample_rate"],
#     duration=CONFIG["duration"],
#     n_fft=CONFIG["n_fft"],
#     hop_length=CONFIG["hop_length"],
#     n_mels=CONFIG["n_mels"],
#     top_db=CONFIG["top_db"],
# )
# logger.debug(f"Test dataset size: {len(test_data)}")

# test_data.mean = dataset.mean
# test_data.std = dataset.std

### 2. Load model

In [ ]:
models = {
    "resnet18": (
        make_resnet18, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1, # TODO: what is this and why is it?
            "small_inputs": True, # TODO: what is this and why is it?
            "conv_layer": partial(BcosConv2d, b=2, max_out=2), # TODO: what is this and why is it 1?
        }
    ),
}
model = None
for name, (cls, kwargs) in models.items():
    if CONFIG['model'].lower() in name:
        model = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model})."

logger.debug(f"Model:\n{model}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model = model.to(DEVICE)


In [ ]:
model_file_path = [entry for entry in os.listdir(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}") if entry.endswith(".pth")][0]

state_dict = torch.load(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/{model_file_path}", weights_only=True)
model.load_state_dict(state_dict)

### 3. create the explanations

In [ ]:
index = 0
SINGLE_TRAIN_IMG = dataset[index][0]
SINGLE_TRAIN_LABEL = dataset[index][1]

plt.imshow(SINGLE_TRAIN_IMG.squeeze(), origin="lower")
plt.title(f"label: {REVERSE_LABEL_MAP[SINGLE_TRAIN_LABEL]}")

In [ ]:
model.eval()
expl_out = model.explain(SINGLE_TRAIN_IMG.unsqueeze(0).to(DEVICE).requires_grad_(), SINGLE_TRAIN_LABEL)

plt.imshow(expl_out["explanation"], origin="lower")
plt.title(f"Prediction: {REVERSE_LABEL_MAP[expl_out["prediction"]]}")
plt.show()

In [ ]:
cm = expl_out["contribution_map"][0].cpu().detach().numpy()

plt.figure(figsize=(8,4))
plt.imshow(cm, origin="lower", cmap="bwr")
plt.title(f"Prediction: {REVERSE_LABEL_MAP[expl_out["prediction"]]}")
plt.colorbar()

In [ ]:
cm = expl_out["contribution_map"][0].cpu().detach().numpy()

logger.info(f"{cm.min() = }, {cm.max() =}")
logger.info(f"{(cm > 0).mean() = }")
logger.info(f"{(cm < 0).mean() = }")

plt.hist(cm.flatten(), bins=100)
plt.show()

In [ ]:
CAT_1, _ = dataset[0]
CAT_2, _ = dataset[3]

DOG_1, _ = dataset[402]
DOG_2, _ = dataset[403]

predict_for_label = "dog"

# --------------------------------------------------
# Build 2x2 grid
#
# CAT | DOG
# ----+----
# CAT | DOG
#
# dog quadrants are top-right and bottom-right
# --------------------------------------------------

def make_grid(a, b, c, d):
    """
    Inputs: tensors (C,H,W)
    Output: tensor (C,2H,2W)
    """

    top = torch.cat([a, b], dim=-1)
    bottom = torch.cat([c, d], dim=-1)

    return torch.cat([top, bottom], dim=-2)

grid_img = make_grid(
    CAT_1,
    DOG_1,
    CAT_2,
    DOG_2,
)

# --------------------------------------------------
# Explain
# --------------------------------------------------

x = (
    grid_img
    .unsqueeze(0)
    .to(DEVICE)
    .requires_grad_()
)

model.eval()

with torch.enable_grad():
    expl_out = model.explain(x, idx=LABEL_MAP[predict_for_label])

cm = expl_out["contribution_map"][0].detach().cpu().numpy()

# --------------------------------------------------
# Grid Pointing Game
# --------------------------------------------------

H, W = cm.shape

peak_y, peak_x = np.unravel_index(
    np.argmax(cm),
    cm.shape
)

# dog quadrants = right half
dog_hit = peak_x >= (W // 2)

logger.info(f"Predicting for: {predict_for_label}")
logger.info(f"Peak location: ({peak_y}, {peak_x})")
logger.info(f"GridPG hit: {dog_hit if predict_for_label == "dog" else not dog_hit}")

# --------------------------------------------------
# Visualisation
# --------------------------------------------------

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

# Composite spectrogram
display_img = grid_img.squeeze().cpu().numpy()

ax[0].imshow(display_img, origin="lower")
ax[0].axvline(W//2, color="white")
ax[0].axhline(H//2, color="white")
ax[0].set_title("Composite Grid")

# Contribution map
im = ax[1].imshow(cm, origin="lower", cmap="bwr")
ax[1].scatter(
    peak_x,
    peak_y,
    marker="x",
    s=200,
    c="black"
)
ax[1].axvline(W//2, color="black")
ax[1].axhline(H//2, color="black")
ax[1].set_title("Contribution Map")

plt.colorbar(im, ax=ax[1])

# Positive contributions only
positive_cm = np.maximum(cm, 0)

ax[2].imshow(
    positive_cm,
    origin="lower",
    cmap="hot"
)
ax[2].scatter(
    peak_x,
    peak_y,
    marker="x",
    s=200,
    c="white"
)
ax[2].axvline(W//2, color="white")
ax[2].axhline(H//2, color="white")
ax[2].set_title("Positive Contributions")

plt.tight_layout()
plt.show()